In [1]:
import json

from pydantic import BaseModel
from pyspark.sql import SparkSession
import numpy as np

In [2]:
class Evaluation(BaseModel):
    TEDS: float
    filename: str
    is_complex: bool
    pred_ncols: int
    pred_nrows: int
    table_id: int
    true_ncols: int
    true_nrows: int
    timing: float

class BenchmarkStats(BaseModel):
    benchmark_name: str
    provider_name: str
    TEDS_mean: float
    TEDS_median: float
    TEDS_std: float
    timings_mean: float
    timings_std: float
    num_evaluations: int
    evaluations: list[Evaluation]

In [3]:
benchmark_name = "OmniDocBench"
provider_name = "cells2table"

In [5]:
with open(f"../benchmarks/{benchmark_name}/{provider_name}/evaluations/evaluation_{benchmark_name}_table_structure.json") as file:
    data = json.load(file)

In [44]:
spark = SparkSession.builder.appName("timings").config("spark.driver.memory", "4g").config("spark.executor.memory", "4g").getOrCreate()

In [ ]:
df = spark.read.parquet(f"../benchmarks/{benchmark_name}/{provider_name}/test/").select(["document_id", "prediction_timings"])

In [46]:
evaluations: list[Evaluation] = []
teds: list[float] = []
timings: list[float] = []

for eval in data["table_structure_evaluations"]:
    if eval["structure_only_evaluation"]:
        row = df.filter(df["document_id"] == eval["filename"]).first()
        if row is None:
            break

        eval["timing"] = json.loads(row["prediction_timings"])["table_structure"][eval["table_id"]]

        timings.append(eval["timing"])
        teds.append(eval["TEDS"])
        evaluations.append(Evaluation.model_validate(eval))

In [47]:
TEDS_mean = np.mean(teds)
TEDS_std = np.std(teds)
TEDS_median = np.median(teds)

timings_mean = np.mean(timings)
timings_std = np.std(timings)

In [ ]:
stats = BenchmarkStats(
    benchmark_name=benchmark_name,
    provider_name=provider_name,
    TEDS_mean=TEDS_mean,  # ty:ignore[invalid-argument-type]
    TEDS_std=TEDS_std,  # ty:ignore[invalid-argument-type]
    TEDS_median=TEDS_median,
    timings_mean=timings_mean,  # ty:ignore[invalid-argument-type]
    timings_std=timings_std,  # ty:ignore[invalid-argument-type]
    num_evaluations=len(evaluations),
    evaluations=evaluations,
)

In [49]:
stats_json = stats.model_dump_json(indent=2)

In [50]:
with open(f"{benchmark_name}_{provider_name}.json", "w") as file:
    file.write(stats_json)